# Project 2: Clustering

## TASK #1: UNDERSTAND THE PROBLEM STATEMENT AND BUSINESS CASE

* Marketing is crucial for the growth and sustainability of any business.
* Marketers can help build the company’s brand, engage customers, grow revenue, and increase sales.
* One of the key pain points for marketers is to know their customers and identify their needs.
* By understanding the customer, marketers can launch a targeted marketing campaign that is tailored for specific needs.
* If data about the customers is available, data science can be applied to perform market segmentation.
* In this case study, we have extensive data of a bank in New York City on their customers for the past 6 months. 
* The marketing team at the bank wants to launch a targeted ad marketing campaign by dividing their customers into at least 3 distinctive groups.  

dataset includes:

o BALANCE<br>
o BALANCE_FREQUENCY<br>
o PURCHASES<br>
o INSTALLMENTS_PURCHASES<br>
o CASH_ADVANCE<br>
o PURCHASES_FREQUENCY<br>
o PURCHASES_INSTALLMENTS_FREQUENCY<br>

Data Source: https://www.kaggle.com/arjunbhasin2013/ccdata

**Skills & techniques:**

- Market segmentation, Dimension reduction
- EDA: distplot, histograms, KDE
- Feature Engineering:
    - Filling missing data
- Models:
    - K-means clustering (elbow method)
    - Autoencoders
    - PCA

## TASK #2: IMPORT LIBRARIES AND DATASETS

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA


In [ ]:
df = pd.read_csv('Marketing_data.csv')
df.head()

**Features:**

CUSTID: Identification of Credit Card holder  
BALANCE: Balance amount left in customer's account to make purchases  
BALANCE_FREQUENCY: How frequently the Balance is updated, score between 0 and 1 (1 = frequently updated, 0 = not frequently updated)  
PURCHASES: Amount of purchases made from account  
ONEOFFPURCHASES: Maximum purchase amount done in one-go  
INSTALLMENTS_PURCHASES: Amount of purchase done in installment  
CASH_ADVANCE: Cash in advance given by the user  
PURCHASES_FREQUENCY: How frequently the Purchases are being made, score between 0 and 1 (1 = frequently purchased, 0 = not frequently purchased)  
ONEOFF_PURCHASES_FREQUENCY: How frequently Purchases are happening in one-go (1 = frequently purchased, 0 = not frequently purchased)  
PURCHASES_INSTALLMENTS_FREQUENCY: How frequently purchases in installments are being done (1 = frequently done, 0 = not frequently done)  
CASH_ADVANCE_FREQUENCY: How frequently the cash in advance being paid  
CASH_ADVANCE_TRX: Number of Transactions made with "Cash in Advance"  
PURCHASES_TRX: Number of purchase transactions made  
CREDIT_LIMIT: Limit of Credit Card for user  
PAYMENTS: Amount of Payment done by user  
MINIMUM_PAYMENTS: Minimum amount of payments made by user  
PRC_FULL_PAYMENT: Percent of full payment paid by user  
TENURE: Tenure of credit card service for user  

In [ ]:
df.info()
df.describe().T

In [ ]:
# Let's see who made one off purchase of ~$40000!
df[df['ONEOFF_PURCHASES'] > 40000].T

In [ ]:
# Let's see who made cash advance of ~$47000!
df[df['CASH_ADVANCE'] > 47000].T
# This customer made 123 cash advance transactions!!
# Never paid credit card in full

## TASK #3: VISUALIZE AND EXPLORE DATASET

In [ ]:
# Let's see if we have any missing data
df.isna().sum()


It should be fine to fill out two columns with the median/mean of others.

In [ ]:
# Fill up the missing elements with mean of the 'MINIMUM_PAYMENT'
df['MINIMUM_PAYMENTS'].fillna(df['MINIMUM_PAYMENTS'].mean(), inplace=True)

In [ ]:
# Fill up the missing elements with mean of the 'CREDIT_LIMIT'
df['CREDIT_LIMIT'].fillna(df['CREDIT_LIMIT'].mean(), inplace=True)

In [ ]:
df.isna().sum()

In [ ]:
# Check if we have duplicated entries in the data
df.duplicated().sum()

In [ ]:
# Let's drop Customer ID since it has no meaning here
df.drop(columns=['CUST_ID'], inplace=True)
df.head()

In [ ]:
print(df.columns)
print(f'# of columns: {df.shape[1]}')

In [ ]:
# distplot combines the matplotlib.hist function with seaborn kdeplot()
# KDE Plot represents the Kernel Density Estimate
# KDE is used for visualizing the Probability Density of a continuous variable.
# KDE demonstrates the probability density at different values in a continuous variable.

# Mean of balance is $1500
# 'Balance_Frequency' for most customers is updated frequently ~1
# For 'PURCHASES_FREQUENCY', there are two distinct group of customers
# For 'ONEOFF_PURCHASES_FREQUENCY' and 'PURCHASES_INSTALLMENT_FREQUENCY' most users don't do one off puchases or installment purchases frequently
# Very small number of customers pay their balance in full 'PRC_FULL_PAYMENT'~0
# Credit limit average is around $4500
# Most customers are ~11 years tenure

fig, axes = plt.subplots(6,3, figsize=(22, 22))
axes = axes.ravel()

for i, column in enumerate(df.columns):
    sns.histplot(df[column], ax=axes[i], kde=True)
    axes[i].set_title(column)

plt.tight_layout()

In [ ]:
sns.pairplot(df)

In [ ]:
corr = df.corr()
print(corr)
plt.figure(figsize=(12, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='YlGnBu')   # from low to high: yellow to green to blue
plt.show()


In [ ]:

# 'PURCHASES' have high correlation between one-off purchases, 'installment purchases, purchase transactions, credit limit and payments.
# Strong Positive Correlation between 'PURCHASES_FREQUENCY' and 'PURCHASES_INSTALLMENT_FREQUENCY'


## TASK #4: FIND THE OPTIMAL NUMBER OF CLUSTERS USING ELBOW METHOD

In [ ]:
# Let's scale the data first
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)
print(df_scaled[:3])

In [ ]:
wcss = []
for i in range(1, 20):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(df_scaled)
    wcss.append(kmeans.inertia_)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, 20), wcss, marker='o', linestyle='--')
plt.title('Elbow Method For Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('WCSS')
plt.xticks(range(1, 20))
plt.show()

In [ ]:

# From this we can observe that, 4th cluster seems to be forming the elbow of the curve.
# However, the values does not reduce linearly until 8th cluster.
# Let's choose the number of clusters to be 7.

## TASK #5: APPLY K-MEANS METHOD

In [ ]:
kmeans = KMeans(n_clusters=8, init='k-means++', random_state=42)
kmeans.fit(df_scaled)
labels = kmeans.labels_
print(labels)

In [ ]:
kmeans.cluster_centers_.shape

In [ ]:
cluster_centers = pd.DataFrame(kmeans.cluster_centers_, columns=df.columns)
cluster_centers

In [ ]:
# In order to understand what these numbers mean, let's perform inverse transformation
cluster_centers_unscaled = scaler.inverse_transform(cluster_centers)
cluster_centers_unscaled_df = pd.DataFrame(cluster_centers_unscaled, columns=df.columns)
cluster_centers_unscaled_df

# 0. First customers cluster (revolvers):
# who use credit card as a loan (most lucrative sector)
# highest balance ($5848) and cash advance (~$8200), relatively low purchase frequency (0.42),
# high cash advance frequency (0.62), high cash advance transactions (25) and low percentage of full payment (0.07%)


# 1. Second Customers cluster (Transactors):
# Those are customers who pay least amount of intrerest charges and careful with their money,
# Cluster with lowest balance ($113) and cash advance ($325), Percentage of full payment = 0.23%


# 4. Third customer cluster (VIP/Prime):
# high credit limit $12400 and highest percentage of full payment,
# target for increase credit limit and increase spending habits


In [ ]:
labels.shape, labels.max(), labels.min()

In [ ]:
# Equivalent to kmeans.fit() + kmeans.labels_, we can do it in one step as:
y_kmeans = kmeans.fit_predict(df_scaled)
y_kmeans

In [ ]:
# concatenate the clusters labels to our original dataframe

df_cluster = pd.concat([df, pd.DataFrame({'cluster':labels})], axis = 1)
df_cluster.head()


In [ ]:
# Plot the histogram of various clusters

for col in df.columns:
    fig, axes = plt.subplots(1, 8, figsize=(35, 5), sharey=True)
    for j in range(8):
        cluster = df_cluster[df_cluster['cluster'] == j]
        axes[j].hist(cluster[col], bins=20, color='steelblue', edgecolor='black')
        axes[j].set_title(f"{col}\nCluster {j}")
    fig.tight_layout()
    plt.show()

## TASK 6: APPLY PRINCIPAL COMPONENT ANALYSIS AND VISUALIZE THE RESULTS

In [ ]:
# Obtain the principal components
pca = PCA(n_components=2)
pca_result = pca.fit_transform(df_scaled)
pca_df = pd.DataFrame(data=pca_result, columns=['PCA1', 'PCA2'])
pca_df['cluster'] = labels
pca_df.head()

In [ ]:
# Visualize the clusters based on the first two principal components
plt.figure(figsize=(10, 7))
sns.scatterplot(data=pca_df, x='PCA1', y='PCA2', hue='cluster', palette='Set2', s=100)
plt.title('Clusters of Customers based on PCA')
plt.show()

## TASK #7: APPLY AUTOENCODERS (PERFORM DIMENSIONALITY REDUCTION USING AUTOENCODERS)

In [ ]:
from tensorflow.keras.layers import (
    Dense, Input, Flatten, Add, Activation, ZeroPadding2D,
    Conv2D, BatchNormalization, MaxPooling2D, Dropout
)
